# DIMER brightness-stationarity audit: three-panel video viewer

Viewer for the stationarity-fix comparison. It loads TWO persisted posterior-predictive
clips for the SAME experimental recording from the Data_Bank Posit tier: the render under
the retired grid brightness chain, and a fixed-model render under the stationary
continuous process. Three panels: EXPERIMENTAL | GRID (retired) | FIXED.

This is a pure viewer -- it needs only `numpy`, `matplotlib`, and `ipywidgets` (no project
package, no `MACHINE_PROFILE`, no ReaDDy). It never runs a simulation; to get a new
trajectory, re-run the engine
(`Script_Bank/Analysis/SRM_AND_SBI_DIMER_ALP_Posterior_Predictive_Video.py`) in the worktree.

**Knobs (next cell; re-run it and the cells below after changing any):**

- `KIND`, `CELL` -- which recording: `"MET-FAB"` or `"MET-INLB"`, cell index.
- `FIXED_LABEL` -- which fixed-model render is the FIXED panel: `"OU_FIX"` (frozen imaging
  vector) or `"OU_FIX_SIGMA_053"` (sigma_pc lowered to 0.53; MET-FAB cell 0 only). Any
  `--run-label` token of a render present in the Data_Bank works.
- `PCTL` -- the upper quantile of the `"percentile"` color window.
- `QUANTILES` -- the rows of the printed per-stack summary table.

The color scaling always puts the three panels on ONE shared window, so identical
intensities map to identical colors. `NORM_MODE` picks the window:
`"percentile"` = shared [min, p`PCTL`] over the three stacks (default; the viewing mode --
`PCTL` only acts here);
`"full"` = shared [min, max] over ALL THREE stacks (verification mode; nothing clipped, so
a handful of extreme pixels sets the cap and the panels go dim);
`"autoscale"` = the displayed frames' shared min/max.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, interact
# Pure viewer: numpy + matplotlib + ipywidgets only.

In [ ]:
# ---- knobs -------------------------------------------------------------------------
KIND = "MET-FAB"           # "MET-FAB" | "MET-INLB"
CELL = 0                   # recording index
FIXED_LABEL = "OU_FIX"     # "OU_FIX" | "OU_FIX_SIGMA_053" | any --run-label token
NORM_MODE = "percentile"   # "percentile" (viewing; caps at PCTL) | "full" (verification; nothing clipped) | "autoscale"
PCTL = 99.99               # upper quantile of the "percentile" color window
QUANTILES = (50, 90, 99, 99.9, 99.99)   # rows of the summary table (min/max always shown)
# --------------------------------------------------------------------------------------

# Both clips live in the Data_Bank Posit tier (adjust for another machine).
PPV_ROOT = ("/media/mars-fias/MARS_Data_Reserve_0/MARS_Data_Bank/Projects/RCL_Agent_Projects/"
            "srm-and-sbi-dimer-alp/Data_Bank/Posit/"
            "SRM_AND_SBI_DIMER_ALP_2S_50FPS_Posterior_Predictive_Video/")

def _stem(label=""):
    token = f"_{label}" if label else ""
    return (f"SRM_AND_SBI_DIMER_ALP_2S_50FPS_MAP_Estimate_SGM_{KIND}_Cell_{CELL}"
            f"_20S{token}_Synthetic_Video.npz")

GRID_CLIP = PPV_ROOT + _stem()               # retired grid-chain render
FIXED_CLIP = PPV_ROOT + _stem(FIXED_LABEL)   # fixed-model render

g = np.load(GRID_CLIP, allow_pickle=False)
f = np.load(FIXED_CLIP, allow_pickle=False)
experimental, grid_synth, fixed_synth = g["experimental"], g["synth"], f["synth"]
assert np.array_equal(f["experimental"], experimental), "clips reference different recordings"
stacks = (experimental, grid_synth, fixed_synth)
titles = ("EXPERIMENTAL", "GRID (retired)", f"FIXED ({FIXED_LABEL})")
n_frames = int(g["n_frames"])
fps = round(1.0 / float(g["frame_time_seconds"]))

print(f"{KIND} cell {CELL} -- per-stack summary (ADU):")
header = "".join(f"{'p' + format(q, 'g'):>9s}" for q in QUANTILES)
print(f"{'':26s}{'min':>9s}{header}{'max':>9s}")
for name, s in zip(titles, stacks):
    row = "".join(f"{np.percentile(s, q):9.0f}" for q in QUANTILES)
    print(f"{name:26s}{s.min():9d}{row}{s.max():9d}")

_FULL = (float(min(s.min() for s in stacks)), float(max(s.max() for s in stacks)))
_PCTL = (_FULL[0], float(max(np.percentile(s, PCTL) for s in stacks)))

def clim(*crops):
    """ONE shared (vmin, vmax) for the three panels."""
    if NORM_MODE == "full":
        return _FULL
    if NORM_MODE == "percentile":
        return _PCTL
    return (float(min(c.min() for c in crops)), float(max(c.max() for c in crops)))

print(f"\nNORM_MODE={NORM_MODE}  shared window: "
      + (str(tuple(round(x) for x in (_FULL if NORM_MODE == 'full' else _PCTL)))
         if NORM_MODE != "autoscale" else "(displayed frames' shared min/max)")
      + (f"  (percentile mode caps at p{PCTL:g} = {round(_PCTL[1])})"
         if NORM_MODE == "percentile" else ""))

In [ ]:
# Scrub frames and zoom a shared region-of-interest into ALL THREE panels together.
H, W = experimental.shape[1], experimental.shape[2]

def _roi(frame2d, cx, cy, zoom):
    half_y, half_x = int(H / (2 * zoom)), int(W / (2 * zoom))
    y0, y1 = max(0, cy - half_y), min(H, cy + half_y)
    x0, x1 = max(0, cx - half_x), min(W, cx + half_x)
    return frame2d[y0:y1, x0:x1], (x0, x1, y0, y1)

def show(frame, cx, cy, zoom):
    crops, ext = [], None
    for s in stacks:
        crop, ext = _roi(s[frame], cx, cy, zoom)
        crops.append(crop)
    vmin, vmax = clim(*crops)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
    for ax, crop, title in zip(axes, crops, titles):
        ax.imshow(crop, cmap="magma", origin="lower", interpolation="none",
                  vmin=vmin, vmax=vmax, extent=ext)
        ax.set_title(f"{title}   frame {frame}/{n_frames - 1}   zoom x{zoom:g}")
    plt.tight_layout(); plt.show()

interact(
    show,
    frame=IntSlider(min=0, max=n_frames - 1, step=1, value=n_frames // 2, description="frame"),
    cx=IntSlider(min=0, max=W - 1, step=1, value=W // 2, description="center x"),
    cy=IntSlider(min=0, max=H - 1, step=1, value=H // 2, description="center y"),
    zoom=FloatSlider(min=1.0, max=8.0, step=0.5, value=1.0, description="zoom"),
);

## Playback (real time)

Renders every `STRIDE`-th frame as an inline HTML5 animation at the acquisition rate
divided by `STRIDE` (STRIDE = 2 keeps 20 s of video at 25 fps). The shared color window
follows `NORM_MODE` above.

In [ ]:
import matplotlib.animation as manim
from IPython.display import HTML

STRIDE = 2
sel = range(0, n_frames, STRIDE)
vmin, vmax = clim(*[s for s in stacks]) if NORM_MODE != "autoscale" else _PCTL

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
images = []
for ax, s, title in zip(axes, stacks, titles):
    im = ax.imshow(s[0], cmap="magma", origin="lower", interpolation="none",
                   vmin=vmin, vmax=vmax)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    images.append(im)
label = axes[0].set_ylabel("t = 0.0 s")
plt.tight_layout()

def update(t):
    for im, s in zip(images, stacks):
        im.set_data(s[t])
    label.set_text(f"t = {t / fps:.1f} s")
    return images

anim = manim.FuncAnimation(fig, update, frames=list(sel), interval=1000 * STRIDE / fps, blit=False)
plt.close(fig)
HTML(anim.to_jshtml(default_mode="loop"))